# 1D CNN Training - Optimized Version## Key Improvements:1. **Clean imports** - unified to `tensorflow.keras`2. **EarlyStopping & ReduceLROnPlateau** - prevent overfitting, auto-reduce LR3. **BatchNormalization** - stabilize training4. **tf.data pipeline** - accelerate data loading5. **Mixed Precision** - speed up GPU training6. **Fixed data leakage** - scaler fitted on train data only7. **Fixed output activation** - removed `relu` from output layer (clips negative predictions)8. **Fixed R2 metric** - accumulates across batches for correct computation

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport pandas as pdimport timeitimport osimport tensorflow as tffrom tensorflow.keras.models import Sequentialfrom tensorflow.keras.layers import (    Dense, Dropout, BatchNormalization,    Conv1D, MaxPooling1D, Flatten)from tensorflow.keras.optimizers import Adamfrom tensorflow.keras.callbacks import (    EarlyStopping, ReduceLROnPlateau, CSVLogger, ModelCheckpoint)from sklearn.preprocessing import MinMaxScalerfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score# Enable Mixed Precision for faster GPU training (if available)tf.keras.mixed_precision.set_global_policy('mixed_float16')print(f'TensorFlow version: {tf.__version__}')print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

## 1. Hyperparameter Configuration

In [ ]:
# ============================================================# HYPERPARAMETERS - Adjust all settings here# ============================================================CONFIG = {    # Data    'data_path': '/content/drive/MyDrive/Research/Project Man/Data1/Out_ML_cm.csv',    'n_features_in': 9,        # Number of input columns (0:9)    'n_features_out': 202,     # Number of output columns (10:212)    'train_ratio': 0.8,    'n_samples': 30000,        # Model architecture    'conv_filters': [128, 64, 64],     # Filters per Conv1D layer    'conv_kernel': [2, 2, 2],          # Kernel sizes    'dense_units': [64, 32],           # Dense layer units    'dropout_rate': 0.2,    'pool_size': 2,        # Training    'epochs': 1000,    'batch_size': 128,    'learning_rate': 0.001,    'patience_es': 50,    'patience_lr': 20,    'min_lr': 1e-6,        # Output    'output_dir': '/content/drive/MyDrive/Research/Project Man/CNN/',    'model_name': 'CNNcm',}os.makedirs(CONFIG['output_dir'], exist_ok=True)

## 2. Load & Preprocess Data

In [ ]:
start = timeit.default_timer()# Load data from CSVdataframe = pd.read_csv(CONFIG['data_path'], header=None)dataset = dataframe.values[:CONFIG['n_samples'], :212]print(f'Dataset shape: {dataset.shape}')# Use separate scalers for input and output# This allows inverse_transform on predictions laterscaler_X = MinMaxScaler()scaler_y = MinMaxScaler()n_in = CONFIG['n_features_in']n_split = int(CONFIG['n_samples'] * CONFIG['train_ratio'])X = dataset[:, :n_in]y = dataset[:, 10:212]# FIX: Fit scaler on training data ONLY to prevent data leakage# (Original code fitted on the entire dataset including test data,#  which means test set statistics leak into the training process)scaler_X.fit(X[:n_split])scaler_y.fit(y[:n_split])X = scaler_X.transform(X)y = scaler_y.transform(y)# Train/Test splitx_train, x_test = X[:n_split], X[n_split:]y_train, y_test = y[:n_split], y[n_split:]# Reshape for model input: (samples, timesteps, features)x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], 1)x_test = x_test.reshape(x_test.shape[0], x_test.shape[1], 1)input_node = x_train.shape[1]output_node = y_train.shape[1]n_timesteps = x_train.shape[2]print(f'x_train: {x_train.shape}, y_train: {y_train.shape}')print(f'x_test:  {x_test.shape},  y_test:  {y_test.shape}')

## 3. Create tf.data Pipeline (faster data loading)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNEbatch_size = CONFIG['batch_size']# Build optimized training dataset:# - shuffle: randomize sample order each epoch# - batch: group samples into batches# - prefetch: prepare next batch while GPU processes current onetrain_ds = (    tf.data.Dataset.from_tensor_slices((x_train, y_train))    .shuffle(buffer_size=len(x_train))    .batch(batch_size)    .prefetch(AUTOTUNE))# Validation dataset (no shuffle needed for evaluation)val_ds = (    tf.data.Dataset.from_tensor_slices((x_test, y_test))    .batch(batch_size)    .prefetch(AUTOTUNE))

## 4. Build Model**Improvements over original:**- Added `BatchNormalization` after Conv1D and Dense layers- Removed `relu` activation from output layer (was clipping negative predictions)- Added `Dropout` for regularization- Output layer uses `float32` (required for mixed precision)

In [ ]:
def build_model(config):    """Build 1D CNN model from config dictionary."""    filters = config['conv_filters']    kernels = config['conv_kernel']    dense_units = config['dense_units']    input_shape = (config['n_features_in'], 1)        model = Sequential(name='CNN1D_optimized')        # Conv1D Layer 1 + BatchNorm    model.add(Conv1D(filters[0], kernel_size=kernels[0], activation='relu',                     input_shape=input_shape))    model.add(BatchNormalization())        # Conv1D Layer 2 + BatchNorm    model.add(Conv1D(filters[1], kernel_size=kernels[1], activation='relu'))    model.add(BatchNormalization())        # Conv1D Layer 3 + BatchNorm + Dropout    model.add(Conv1D(filters[2], kernel_size=kernels[2], activation='relu'))    model.add(BatchNormalization())    model.add(Dropout(config['dropout_rate']))        # Pooling and Flatten    model.add(MaxPooling1D(pool_size=config['pool_size']))    model.add(Flatten())        # Dense hidden layers    for u in dense_units:        model.add(Dense(u, activation='relu'))        model.add(BatchNormalization())        # Output layer - no activation for regression    # FIX: Original used relu on output which clips negative values    # dtype must be float32 when using mixed precision    model.add(Dense(config['n_features_out'], dtype='float32'))        return modelmodel = build_model(CONFIG)model.summary()

## 5. Custom R2 Metric

In [ ]:
# Custom R2 metric compatible with mixed precision training## FIX: The original r2_score function computed mean(y_true) per batch,# which gives incorrect R2 values because the global mean differs from# batch-level means. This class accumulates sum, sum-of-squares, and# count across all batches, then computes the correct R2 at epoch end.class R2Score(tf.keras.metrics.Metric):    def __init__(self, name='r2_score', **kwargs):        super().__init__(name=name, **kwargs)        self.ss_res = self.add_weight(name='ss_res', initializer='zeros')        self.y_sum = self.add_weight(name='y_sum', initializer='zeros')        self.y_sq_sum = self.add_weight(name='y_sq_sum', initializer='zeros')        self.n = self.add_weight(name='n', initializer='zeros')        def update_state(self, y_true, y_pred, sample_weight=None):        # Cast to float32 for numerical stability with mixed precision        y_true = tf.cast(y_true, tf.float32)        y_pred = tf.cast(y_pred, tf.float32)        # Accumulate residual sum of squares        self.ss_res.assign_add(tf.reduce_sum(tf.square(y_true - y_pred)))        # Accumulate statistics for total sum of squares        self.y_sum.assign_add(tf.reduce_sum(y_true))        self.y_sq_sum.assign_add(tf.reduce_sum(tf.square(y_true)))        self.n.assign_add(tf.cast(tf.size(y_true), tf.float32))        def result(self):        # R2 = 1 - SS_res / SS_tot        # SS_tot = sum(y^2) - n * mean(y)^2 (computed from accumulated stats)        y_mean = self.y_sum / (self.n + tf.keras.backend.epsilon())        ss_tot = self.y_sq_sum - self.n * tf.square(y_mean)        return 1.0 - self.ss_res / (ss_tot + tf.keras.backend.epsilon())        def reset_state(self):        # Reset all accumulators at the start of each epoch        self.ss_res.assign(0.0)        self.y_sum.assign(0.0)        self.y_sq_sum.assign(0.0)        self.n.assign(0.0)

## 6. Compile & Train

In [ ]:
# Define callbacks for training controlcallbacks = [    # Stop training early if validation loss doesn't improve for patience epochs    EarlyStopping(        monitor='val_loss',        patience=CONFIG['patience_es'],        restore_best_weights=True,  # Restore weights from the best epoch        verbose=1    ),    # Halve learning rate when validation loss plateaus    ReduceLROnPlateau(        monitor='val_loss',        factor=0.5,                 # Multiply LR by 0.5        patience=CONFIG['patience_lr'],        min_lr=CONFIG['min_lr'],    # Don't reduce below this value        verbose=1    ),    # Save the best model checkpoint based on lowest validation loss    ModelCheckpoint(        filepath=os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_best.keras'),        monitor='val_loss',        save_best_only=True,        verbose=1    ),    # Log training history to CSV file for later analysis    CSVLogger(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_history.csv')),]# Compile model with Adam optimizer and MSE lossoptimizer = Adam(learning_rate=CONFIG['learning_rate'])model.compile(    loss='mse',    optimizer=optimizer,    metrics=[R2Score()])# Train the model using tf.data pipelinehistory = model.fit(    train_ds,    validation_data=val_ds,    epochs=CONFIG['epochs'],    callbacks=callbacks,    verbose=1)

## 7. Evaluate Model

In [ ]:
# Evaluate on validation setscores = model.evaluate(val_ds, verbose=1)# Generate predictions for both train and test setspred_train = model.predict(x_train, batch_size=CONFIG['batch_size'])pred_test = model.predict(x_test, batch_size=CONFIG['batch_size'])# Compute metrics using sklearn (more accurate than batch-wise Keras metrics)metrics = {    'mse_train': mean_squared_error(y_train, pred_train),    'mse_test': mean_squared_error(y_test, pred_test),    'mae_train': mean_absolute_error(y_train, pred_train),    'mae_test': mean_absolute_error(y_test, pred_test),    'r2_train': r2_score(y_train, pred_train),    'r2_test': r2_score(y_test, pred_test),}# Record total computing timestop = timeit.default_timer()metrics['computing_time'] = stop - start# Print all metricsfor k, v in metrics.items():    print(f'{k}: {v}')print(f'\nModel scores: {scores}')

## 8. Save Results

In [ ]:
# Save complete model (architecture + weights + optimizer state)model.save(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_final.keras'))# Save model architecture as JSON (for loading structure without weights)json_string = model.to_json()with open(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '.json'), 'w') as f:    f.write(json_string)# Save all evaluation metrics to a text fileoutput_txt = os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_metrics.txt')with open(output_txt, 'w') as f:    for k, v in metrics.items():        f.write(f'{k}: {v}\n')    f.write(f'scores: {scores}\n')    f.write(f'{model.metrics_names[1]}: {scores[1]*100:.2f}%\n')print('All results saved successfully.')

## 9. Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Plot 1: Training & Validation Loss (log scale for better visibility)axes[0].plot(history.history['loss'], label='Train')axes[0].plot(history.history['val_loss'], label='Validation')axes[0].set_title('Loss (MSE)')axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].legend()axes[0].set_yscale('log')# Plot 2: Training & Validation R2 Scoreaxes[1].plot(history.history['r2_score'], label='Train')axes[1].plot(history.history['val_r2_score'], label='Validation')axes[1].set_title('R² Score')axes[1].set_xlabel('Epoch')axes[1].set_ylabel('R²')axes[1].legend()# Plot 3: Learning Rate schedule (recorded by ReduceLROnPlateau callback)if 'lr' in history.history:    axes[2].plot(history.history['lr'])    axes[2].set_title('Learning Rate')    axes[2].set_xlabel('Epoch')    axes[2].set_ylabel('LR')    axes[2].set_yscale('log')else:    axes[2].text(0.5, 0.5, 'LR not recorded',                  ha='center', va='center', transform=axes[2].transAxes)plt.tight_layout()plt.savefig(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_curves.png'), dpi=150)plt.show()

## 10. Prediction vs Ground Truth

In [ ]:
# Randomly select test samples to visualize prediction qualityn_show = 5fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show))for i in range(n_show):    idx = np.random.randint(0, len(x_test))    axes[i].plot(y_test[idx], label='Ground Truth', alpha=0.8)    axes[i].plot(pred_test[idx], label='Prediction', alpha=0.8)    axes[i].set_title(f'Test Sample #{idx}')    axes[i].legend()plt.tight_layout()plt.savefig(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_predictions.png'), dpi=150)plt.show()